# GP Dictionary Builder & Updater

## Objective

This notebook builds and maintains the master Global Parent (GP) dictionary.

The dictionary is the source of truth used to homologate different spellings of the same company into a single canonical Global Parent.

Example

ABC INC
ABC LLC
ABC CORPORATION

↓

ABC

The notebook supports two execution modes.

### CREATE

Build the first GP dictionary from:

- normalization_pairs.csv
- reviewed algorithm output

### UPDATE

Safely update an existing GP dictionary using:

- existing GP dictionary
- reviewed algorithm output

The dictionary is never rebuilt after it exists.

Every execution either:

- creates the initial dictionary
- or appends new validated mappings.

Before saving, the notebook automatically checks for mapping conflicts to ensure dictionary integrity.

In [96]:
from __future__ import annotations
from enum import Enum
from pathlib import Path
import pandas as pd

In [97]:
# ============================================================
# CONFIGURATION
# ============================================================

class ExecutionMode(Enum):
    CREATE = "create"
    UPDATE = "update"

NORMALIZATION_PAIRS_PATH = Path(r"C:\Users\reyiv\Documents\Quaker Houghton\Codigos\limpieza-brandname rep\limpieza-brandname\GP Cleaning\Output\normalization_pairs.csv")

REVIEW_PATH = Path(r"C:\Users\reyiv\Documents\Quaker Houghton\Codigos\limpieza-brandname rep\limpieza-brandname\GP Cleaning\fuzzy_reviewed.csv")

DICTIONARY_PATH = Path(r"C:\Users\reyiv\Documents\Quaker Houghton\Codigos\limpieza-brandname rep\limpieza-brandname\GP Cleaning\Output\gp_dictionary.csv")

In [98]:
class DictionaryUpdatePipeline:
    """
    Build and maintain the Global Parent dictionary.
    """

    def __init__(
        self,
        normalization_pairs_path: Path,
        review_path: Path,
        dictionary_path: Path,
    ) -> None:

        if dictionary_path.is_file():
            self.mode = ExecutionMode.UPDATE
        else:
            self.mode = ExecutionMode.CREATE

        self.normalization_pairs_path = normalization_pairs_path
        self.review_path = review_path
        self.dictionary_path = dictionary_path
        print(f"Execution mode detected: {self.mode.value}")

        # Input DataFrames

        self.normalization_df = pd.DataFrame()

        self.review_df = pd.DataFrame()

        self.dictionary_df = pd.DataFrame()

        # Mapping DataFrames

        self.normalization_mappings_df = pd.DataFrame()

        self.review_mappings_df = pd.DataFrame()

        self.new_mappings_df = pd.DataFrame()

        # Output

        self.dictionary_result_df = pd.DataFrame()

        self.statistics = {
            "normalization_mappings": 0,
            "review_mappings": 0,
            "new_rows": 0,
            "conflicts": 0,
    }
        



    # ============================================================
    # INPUT LOADING
    # ============================================================

    def _load_inputs(self) -> None:
        """
        Load all required inputs depending on the execution mode.
        """

        self._load_review()

        if self.mode == ExecutionMode.CREATE:
            self._load_normalization_pairs()

        else:
            self._load_dictionary()


    def _load_normalization_pairs(self) -> None:
        """
        Load the normalization pairs generated by Notebook 1.
        """

        self.normalization_df = pd.read_csv(
            self.normalization_pairs_path
        )


    def _load_review(self) -> None:
        """
        Load the manually reviewed algorithm output.
        """

        self.review_df = pd.read_csv(
            self.review_path
        )


    def _load_dictionary(self) -> None:
        """
        Load the existing GP dictionary.
        """

        self.dictionary_df = pd.read_csv(
            self.dictionary_path
        )




    # ============================================================
    # INPUT VALIDATION
    # ============================================================

    def _validate_inputs(self) -> None:
        """
        Validate all input DataFrames required for the selected mode.
        """

        self._validate_review()

        if self.mode == ExecutionMode.CREATE:
            self._validate_normalization_pairs()

        else:
            self._validate_dictionary()


    def _validate_review(self) -> None:
        """
        Validate the reviewed algorithm output.
        """

        required_columns = [
            "GP",
            "Candidate",
            "Score",
            "Decision",
        ]

        missing_columns = [
            column
            for column in required_columns
            if column not in self.review_df.columns
        ]

        if missing_columns:
            raise ValueError(
                f"Review file is missing columns: {missing_columns}"
            )


    def _validate_normalization_pairs(self) -> None:
        """
        Validate the normalization pairs.
        """

        required_columns = [
            "Global Parent",
            "Normalized GP",
        ]

        missing_columns = [
            column
            for column in required_columns
            if column not in self.normalization_df.columns
        ]

        if missing_columns:
            raise ValueError(
                f"Normalization file is missing columns: {missing_columns}"
            )


    def _validate_dictionary(self) -> None:
        """
        Validate the existing GP dictionary.
        """

        required_columns = [
            "Original",
            "Canonical",
        ]

        missing_columns = [
            column
            for column in required_columns
            if column not in self.dictionary_df.columns
        ]

        if missing_columns:
            raise ValueError(
                f"Dictionary is missing columns: {missing_columns}"
            )



    # ============================================================
    # REVIEW MAPPINGS
    # ============================================================

    def _build_review_mappings(self) -> None:
        """
        Convert reviewed homologations into dictionary mappings.
        """

        reviewed = self.review_df.copy()

        reviewed = reviewed[
            reviewed["Decision"].notna()
        ]

        reviewed = reviewed[
            reviewed["Decision"].astype(str).str.strip() != ""
        ]

        mappings = []

        for _, row in reviewed.iterrows():

            gp = str(row["GP"]).strip()
            candidate = str(row["Candidate"]).strip()
            canonical = str(row["Decision"]).strip()

            # GP -> Canonical
            mappings.append(
                {
                    "Original": gp,
                    "Canonical": canonical,
                }
            )

            # Candidate -> Canonical
            mappings.append(
                {
                    "Original": candidate,
                    "Canonical": canonical,
                }
            )

        self.review_mappings_df = (
            pd.DataFrame(mappings)
            .drop_duplicates(
                subset=["Original", "Canonical"]
            )
            .reset_index(drop=True)
        )

        self.statistics["review_mappings"] = len(
            self.review_mappings_df
        )




    # ============================================================
    # DICTIONARY CREATION
    # ============================================================

    def _create_dictionary(self) -> None:
        """
        Create the initial dictionary from normalization mappings
        and reviewed homologations.
        """

        self.dictionary_result_df = (
            pd.concat(
                [
                    self.normalization_mappings_df,
                    self.review_mappings_df,
                ],
                ignore_index=True,
            )
            .drop_duplicates(
                subset=["Original", "Canonical"]
            )
            .reset_index(drop=True)
        )

        self.statistics["new_rows"] = len(
            self.dictionary_result_df
        )



    # ============================================================
    # DICTIONARY UPDATE
    # ============================================================

    def _update_dictionary(self) -> None:
        """
        Append new reviewed mappings to the existing dictionary.
        """

        self.dictionary_result_df = (
            pd.concat(
                [
                    self.dictionary_df,
                    self.review_mappings_df,
                ],
                ignore_index=True,
            )
            .drop_duplicates(
                subset=["Original", "Canonical"]
            )
            .reset_index(drop=True)
        )

        self.statistics["new_rows"] = (
            len(self.dictionary_result_df)
            - len(self.dictionary_df)
        )




    # ============================================================
    # CONFLICT DETECTION
    # ============================================================

    def _detect_conflicts(self) -> None:
        """
        Detect Original values that map to multiple Canonicals.
        """

        conflicts = (
            self.dictionary_result_df
            .groupby("Original")["Canonical"]
            .nunique()
        )

        conflicts = conflicts[
            conflicts > 1
        ]

        if conflicts.empty:
            return

        conflict_df = self.dictionary_result_df[
            self.dictionary_result_df["Original"].isin(conflicts.index)
        ].sort_values(
            by=["Original", "Canonical"]
        )

        self.statistics["conflicts"] = len(conflicts)

        print("\nConflicting mappings detected:\n")

        display(conflict_df)

        raise ValueError(
            f"{len(conflicts)} conflicting Original values were found."
        )


    # ============================================================
    # SORTING
    # ============================================================

    def _sort_dictionary(self) -> None:
        """
        Sort the dictionary by Canonical and Original.
        """

        self.dictionary_result_df = (
            self.dictionary_result_df
            .sort_values(
                by=[
                    "Canonical",
                    "Original",
                ]
            )
            .reset_index(drop=True)
        )


    # ============================================================
    # SAVE
    # ============================================================

    def _save_dictionary(self) -> None:
        """
        Save the GP dictionary.
        """

        self.dictionary_result_df.to_csv(
            self.dictionary_path,
            index=False,
        )


    # ============================================================
    # SUMMARY
    # ============================================================

    def _print_summary(self) -> None:
        """
        Print execution summary.
        """

        print("\nExecution completed successfully.\n")

        print(f"Mode: {self.mode.value}")

        print(f"Normalization mappings: {self.statistics['normalization_mappings']}")

        print(f"Review mappings: {self.statistics['review_mappings']}")

        print(f"New dictionary rows: {self.statistics['new_rows']}")

        print(f"Conflicts detected : {self.statistics['conflicts']}")

        print(f"Final dictionary size: {len(self.dictionary_result_df)}")


    def _build_normalization_mappings(self):

        self.normalization_mappings_df = (
            self.normalization_df
            .rename(
                columns={
                    "Global Parent": "Original",
                    "Normalized GP": "Canonical",
                }
            )[["Original", "Canonical"]]
        )

        # Remove identity mappings
        self.normalization_mappings_df = (
            self.normalization_mappings_df[
                self.normalization_mappings_df["Original"]
                != self.normalization_mappings_df["Canonical"]
            ]
            .drop_duplicates()
            .reset_index(drop=True)
        )

        self.statistics["normalization_mappings"] = len(
        self.normalization_mappings_df
        )


    def run(self):

        self._load_inputs()

        self._validate_inputs()

        if self.mode == ExecutionMode.CREATE:

            self._build_normalization_mappings()

            self._build_review_mappings()

            self._create_dictionary()

        else:

            self._build_review_mappings()

            self._update_dictionary()

        self._detect_conflicts()

        self._sort_dictionary()

        self._save_dictionary()

        self._print_summary()

In [99]:
pipeline = DictionaryUpdatePipeline(
    normalization_pairs_path=NORMALIZATION_PAIRS_PATH,
    review_path=REVIEW_PATH,
    dictionary_path=DICTIONARY_PATH,
)

Execution mode detected: create


In [100]:
pipeline.run()


Execution completed successfully.

Mode: create
Normalization mappings: 2649
Review mappings: 24
New dictionary rows: 2673
Conflicts detected : 0
Final dictionary size: 2673
